# Text → Gloss tiếng Việt: chạy suy luận từ 2 file zip

## 1. Chuẩn bị thư mục
Để `inference.ipynb`, `best_model_export.zip` và `results_bundle.zip` chung một thư mục. Tạo sẵn 2 thư mục rồi giải nén **vào trong** chúng (file trong zip nằm thẳng ở gốc, không có thư mục bọc ngoài):

```
thu_muc_lam_viec/
├── inference.ipynb
├── best_model_export.zip
├── results_bundle.zip
├── best_model/          <- giải nén best_model_export.zip vào đây
│   ├── model.safetensors
│   ├── config.json
│   ├── canon_map.json   (bắt buộc: khôi phục ký hiệu gốc như "có … không")
│   └── spiece.model, tokenizer.json, ...
└── results_bundle/      <- giải nén results_bundle.zip vào đây
    └── processed_data/
        └── gloss_vocab.json   (dùng để kiểm tra unknown_signs)
```

## 2. Cài thư viện (chạy 1 lần trong terminal, rồi khởi động lại kernel)
```
pip install "transformers>=4.50,<5" torch sentencepiece # nếu chạy local
!pip -q install "transformers>=4.50,<5" # nếu chạy GG Colab/Kaggle
```
Phải ghim `transformers` dưới bản 5. Bản 5.x gây lỗi `'dict' object is not an instance of 'Sequence'` khi nạp tokenizer.

## 3. Sửa 2 đường dẫn ở cell code bên dưới
```python
MODEL_DIR = "./best_model"
VOCAB     = "./results_bundle/processed_data/gloss_vocab.json"
```

## 4. Chạy và đọc kết quả
Gọi `predict_json(["câu 1", "câu 2"])`. Mỗi câu trả về:
- `text`: câu sau khi chuẩn hóa (hạ chữ thường).
- `Gloss`: danh sách ký hiệu.
- `unknown_signs`: ký hiệu model sinh ra nhưng **không có** trong kho. Danh sách này rỗng chưa chắc là đúng, vì model vẫn có thể thay bằng một ký hiệu có sẵn nhưng sai nghĩa.

Ví dụ đúng: `em có thích quả táo không?` → `["em", "thích", "quả táo", "có … không"]`.

## 5. Lưu ý
- Không cần GPU. Code tự dùng `cuda` nếu có, không thì dùng CPU (chậm hơn). Giảm `num_beams` xuống 1 hoặc 2 để chạy nhanh hơn.
- Truyền cả list câu vào một lần gọi sẽ nhanh hơn gọi từng câu.
- Nếu ra `có .... không` (bốn dấu chấm) hoặc báo lỗi khi nạp `canon_map.json`, kiểm tra lại `MODEL_DIR` có đúng thư mục chứa `canon_map.json` không.
- `PREFIX` và `MAX_LEN` trong code phải giữ nguyên như lúc train (`"dịch sang gloss: "` và `128`).

In [1]:
import json, re, unicodedata, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ====== SỬA 2 DÒNG NÀY ======
MODEL_DIR = "./best_model_export"        # máy local vd: "./best_model"
VOCAB     = "./results_bundle/processed_data/gloss_vocab.json"  # máy local vd: "./gloss_vocab.json"
# ============================
PREFIX, MAX_LEN = "dịch sang gloss: ", 128   # phải đúng như lúc train

tok   = AutoTokenizer.from_pretrained(MODEL_DIR)
dev   = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(dev).eval()
CANON = json.load(open(f"{MODEL_DIR}/canon_map.json", encoding="utf-8"))
ALL_SIGNS = set(json.load(open(VOCAB, encoding="utf-8")))

nfc = lambda s: re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()
canon_tok = lambda t: CANON.get(t, t.replace("_", " "))

def predict_json(sentences, num_beams=4):
    if isinstance(sentences, str): sentences = [sentences]
    clean = [nfc(nfc(x).lower()) for x in sentences]      # train không có chữ hoa
    inp = tok([PREFIX + c for c in clean], return_tensors="pt", padding=True,
              truncation=True, max_length=MAX_LEN).to(dev)
    with torch.no_grad():
        o = model.generate(**inp, num_beams=num_beams, max_length=MAX_LEN)
    glosses = [[canon_tok(t) for t in d.split()]
               for d in tok.batch_decode(o, skip_special_tokens=True)]
    return [{"text": t, "Gloss": g, "unknown_signs": [x for x in g if x not in ALL_SIGNS]}
            for t, g in zip(clean, glosses)]

print(json.dumps(predict_json(["Em có thích quả táo không?",
                               "Hôm nay tôi cảm thấy hoảng sợ"]),
                 ensure_ascii=False, indent=1))

[
 {
  "text": "em có thích quả táo không?",
  "Gloss": [
   "em",
   "thích",
   "quả táo",
   "có … không"
  ],
  "unknown_signs": []
 },
 {
  "text": "hôm nay tôi cảm thấy hoảng sợ",
  "Gloss": [
   "hôm nay",
   "tôi",
   "cảm thấy",
   "hoảng sợ"
  ],
  "unknown_signs": []
 }
]


In [14]:
print(json.dumps(predict_json(["tôi không thông minh",
                               "tôi đẹp không"]),
                 ensure_ascii=False, indent=1))

[
 {
  "text": "tôi không thông minh",
  "Gloss": [
   "tôi",
   "thông minh",
   "không có"
  ],
  "unknown_signs": []
 },
 {
  "text": "tôi đẹp không",
  "Gloss": [
   "tôi",
   "đẹp không (người)"
  ],
  "unknown_signs": []
 }
]
